In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [3]:
df = pd.read_csv("../data/processed/CVD_cleaned.csv")

In [4]:
features = [
    "Age_Category",
    "General_Health",
    "Diabetes",
    "Skin_Cancer",
    "Other_Cancer",
    "Depression",
    "Smoking_History",
    "Exercise",
    "Arthritis",
    "Sex",
    "Alcohol_Consumption",
    "Fruit_Consumption",
    "Green_Vegetables_Consumption",
    "FriedPotato_Consumption",
    "BMI"
]

In [5]:
X = df[features]
Y = df["Heart_Disease"].map({"No": 0, "Yes": 1})

In [6]:
numerical_cols = ["Alcohol_Consumption","Fruit_Consumption","Green_Vegetables_Consumption","BMI","FriedPotato_Consumption"]
categorical_cols = [
    "Diabetes",
    "Skin_Cancer",
    "Other_Cancer",
    "Depression",
    "Smoking_History",
    "Exercise",
    "Arthritis",
    "Sex",
    ]
ordinal_cols = ["General_Health", "Age_Category"]
X["Age_Category"].unique()

<StringArray>
['70-74', '60-64', '75-79',   '80+', '65-69', '50-54', '45-49', '18-24',
 '30-34', '55-59', '35-39', '40-44', '25-29']
Length: 13, dtype: str

In [7]:
# Ordinal order
general_health_order = ["Poor","Fair","Good","Very Good","Excellent"]
age_order = ['18-24', '25-29', '30-34', '35-39','40-44','45-49','50-54','55-59','60-64','65-69','70-74', '75-79','80+']

In [8]:
ordinal_encoder = OrdinalEncoder(categories=[general_health_order,age_order])

In [9]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("ord",ordinal_encoder, ordinal_cols),
    ("cat", OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),categorical_cols)
])

## Model 1: Logistic Regression

In [10]:
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

In [11]:
pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model",model)
]
)

In [12]:
x_train, x_test, y_train, y_test = train_test_split(
    X,Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

In [13]:
pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [14]:
y_pred = pipeline.predict(x_test)
y_prob = pipeline.predict_proba(x_test)[:,1]
confusion_matrix(y_test, y_pred)

array([[41390, 15207],
       [ 1037,  3941]])

In [15]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.73      0.84     56597
           1       0.21      0.79      0.33      4978

    accuracy                           0.74     61575
   macro avg       0.59      0.76      0.58     61575
weighted avg       0.91      0.74      0.79     61575



In [16]:
roc_auc_score(y_test,y_prob)

0.8351328597565244

Recall is prioritized over precision to minimize false negatives, ensuring the most heart disease cases are detected. However, efforts will be made to improve precision to reduce unnecessary false alarms.

### Observations
- Recall = 79%, out of all patients who actually have heart disease, 79% were correctly detected.
- Precison = 21%, The model is overwarning the patient.

In [17]:
joblib.dump(pipeline, "../models/baseline_model.pkl")

['../models/baseline_model.pkl']

## Tuning the baseline model

In [18]:
from sklearn.metrics import precision_recall_curve

In [19]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
target_recall = 0.80
acceptable_indices = np.where(recalls >= target_recall)[0]
best_index = acceptable_indices[np.argmax(precisions[acceptable_indices])]
best_threshold = thresholds[min(best_index, len(thresholds) - 1)]
print(f"Optimal Threshold: {best_threshold:.3f}")
print(f"Expected Precision: {precisions[best_index]:.3f}")
print(f"Expected Recall: {recalls[best_index]:.3f}")

Optimal Threshold: 0.493
Expected Precision: 0.204
Expected Recall: 0.800


In [20]:
y_pred_tuned = (y_prob >= best_threshold).astype(int)
confusion_matrix(y_test, y_pred_tuned)

array([[41049, 15548],
       [  995,  3983]])

In [21]:
print(classification_report(y_test, y_pred_tuned))

              precision    recall  f1-score   support

           0       0.98      0.73      0.83     56597
           1       0.20      0.80      0.33      4978

    accuracy                           0.73     61575
   macro avg       0.59      0.76      0.58     61575
weighted avg       0.91      0.73      0.79     61575



### Model Evaluation Conclusion:
- The logistic regression model achieved good recall (~0.80) but suffered a very low precision (~0.20), indicating a high number of false positives. Threshold tuning provided only marginal improvement, showing the model's limited ability to capture complex patterns in the data.
- Therefore, Logistic regression model is not suitable as the final model, and more powerful models such as tree-based algorithms will be explored.

## Model 2: Random Forest Classifier
* To address the severe imbalance in the dataset, a balanced random forest classifier was implemented using the imblearn library.
* The goal is to improve the model's ability to correctly identify heart disease case.

In [22]:
from imblearn.ensemble import BalancedRandomForestClassifier

In [23]:
brfc_model = BalancedRandomForestClassifier(
    n_estimators=100,
    sampling_strategy='auto',
    replacement=True,
    random_state=42,
)

In [24]:
brfc_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",brfc_model)
])

In [25]:
brfc_pipeline.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [26]:
y_pred = brfc_pipeline.predict(x_test)
y_prob = brfc_pipeline.predict_proba(x_test)[:,1]

In [27]:
confusion_matrix(y_test,y_pred)

array([[39542, 17055],
       [  944,  4034]])

In [28]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.70      0.81     56597
           1       0.19      0.81      0.31      4978

    accuracy                           0.71     61575
   macro avg       0.58      0.75      0.56     61575
weighted avg       0.91      0.71      0.77     61575



In [29]:
roc_auc_score(y_test,y_prob)

0.824117230182824

### Tuning Balanced Random Forest Classifier

In [30]:
from sklearn.model_selection import GridSearchCV

In [31]:
rmodel = BalancedRandomForestClassifier(random_state=42)

In [32]:
pipeliner = Pipeline([
    ("preprocessor",preprocessor),
    ("model",rmodel)
])

In [33]:
param_grid = {
    "model__n_estimators":[100,200],
    "model__max_depth": [5,10,15,20],
    "model__min_samples_leaf":[2,5,7,8,9],
}


In [34]:
grid = GridSearchCV(
    estimator=pipeliner,
    param_grid=param_grid,
    scoring='roc_auc',
    cv = 3,
    n_jobs=-1,
    verbose=2
)
grid.fit(x_train,y_train)

Fitting 3 folds for each of 40 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [5, 10, ...], 'model__min_samples_leaf': [2, 5, ...], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidat

In [35]:
best_modell = grid.best_estimator_
print(f"Best Performance found: {grid.best_params_}")

Best Performance found: {'model__max_depth': 10, 'model__min_samples_leaf': 9, 'model__n_estimators': 200}


In [36]:
y_pred = best_modell.predict(x_test)
y_prob = best_modell.predict_proba(x_test)[:,1]


In [37]:
confusion_matrix(y_test,y_pred)

array([[40286, 16311],
       [  928,  4050]])

In [38]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.71      0.82     56597
           1       0.20      0.81      0.32      4978

    accuracy                           0.72     61575
   macro avg       0.59      0.76      0.57     61575
weighted avg       0.91      0.72      0.78     61575



### Model Evaluation Conclusion:
* Recall: ~0.81
* Precision: ~0.20
* ROC AUC: ~0.81


### Observation:
* The model successfully maintained high recall but remained low.
* Overall performance was comparable to Logistic regression with slightly lower ROC-AUC score

## Model 2: XGBoost Classifier

### Objective: 
To evaluate whether a boosting-based model can outperform previous models in handling class imbalance and improving predictive performance

In [39]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators = 200,
    max_depth = 5,
    learning_rate = 0.1,
    scale_pos_weight = 10,
    random_state=42,
    n_jobs=-1
)
xgb_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model",xgb_model)
])
xgb_pipeline.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [45]:
y__pred = xgb_pipeline.predict(x_test)
y__prob = xgb_pipeline.predict_proba(x_test)[:,1]

In [46]:
confusion_matrix(y_test,y__pred)

array([[42355, 14242],
       [ 1109,  3869]])

In [48]:
print(classification_report(y_test,y__pred))

              precision    recall  f1-score   support

           0       0.97      0.75      0.85     56597
           1       0.21      0.78      0.34      4978

    accuracy                           0.75     61575
   macro avg       0.59      0.76      0.59     61575
weighted avg       0.91      0.75      0.81     61575



In [49]:
roc_auc_score(y_test,y__prob)

0.835288137746186

### Model Evalutation Conclusion: 
* Recall = 0.78, Precision = 0.21
* The model achieved results similar to logistic regression and balanced random forest.
* No significant improvement in recall or precision was observed and roc-auc score remained comparable.
### Conclusion: 
* Despite being more advanced model, XGboost did not outperform simpler models. This suggests that the datatset may not contain strong non-linear patterns that this algorithm can exploit.